In [1]:
# Import libraries
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings("ignore")

# ML
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import SGDClassifier, LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
# Load Dataset
df = pd.read_csv("../datasets/diseases_and_symptoms.csv")

# Make columns lowercase
df.columns = df.columns.str.lower()

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (246945, 378)


,diseases,anxiety and nervousness,depression,shortness of breath,depressive or psychotic symptoms,sharp chest pain,dizziness,insomnia,abnormal involuntary movements,chest tightness,...,stuttering or stammering,problems with orgasm,nose deformity,lump over jaw,sore in nose,hip weakness,back swelling,ankle stiffness or tightness,ankle weakness,neck weakness
0,panic disorder,1,0,1,1,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
1,panic disorder,0,0,1,1,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
2,panic disorder,1,1,1,1,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
3,panic disorder,1,0,0,1,0,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0
4,panic disorder,1,1,0,0,0,0,1,1,1,...,0,0,0,0,0,0,0,0,0,0


In [3]:
# Basic Data Check
print("Total missing values:", df.isnull().sum().sum())

print("Total unique diseases:", df["diseases"].nunique())

print("\nTop 10 diseases:")
print(df["diseases"].value_counts().head(10))

Total missing values: 0
Total unique diseases: 773

Top 10 diseases:
diseases
cystitis                          1219
vulvodynia                        1218
nose disorder                     1218
complex regional pain syndrome    1217
spondylosis                       1216
hypoglycemia                      1215
peripheral nerve disorder         1215
esophagitis                       1215
vaginal cyst                      1215
conjunctivitis due to allergy     1215
Name: count, dtype: int64


In [4]:
# Remove Rare Disease
disease_counts = df["diseases"].value_counts()

valid_diseases = disease_counts[disease_counts >= 5].index

df = df[df["diseases"].isin(valid_diseases)]

print("Filtered dataset shape:", df.shape)
print("Remaining diseases:", df["diseases"].nunique())

Filtered dataset shape: (246823, 378)
Remaining diseases: 721


In [5]:
# SPLIT FEATURES & TARGET
X = df.drop("diseases", axis=1)
y = df["diseases"]

# Convert Yes/No if present
X = X.replace({"Yes": 1, "No": 0})

# Convert to smaller datatype (IMPORTANT for memory)
X = X.astype("int8")

print("Feature shape:", X.shape)

Feature shape: (246823, 377)


In [6]:
# TRAIN-TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (197458, 377)
Test shape: (49365, 377)


In [7]:
# Train model
model = LogisticRegression(max_iter=5000)

model.fit(X_train, y_train)

LogisticRegression(max_iter=5000)

In [8]:
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.866362807657247
                                                          precision    recall  f1-score   support

                               abdominal aortic aneurysm       0.90      1.00      0.95        28
                                        abdominal hernia       0.90      0.99      0.94        81
                                         abscess of nose       0.95      0.93      0.94        58
                                     abscess of the lung       1.00      0.75      0.86         4
                                  abscess of the pharynx       0.85      0.97      0.90        68
                                    acanthosis nigricans       0.56      0.83      0.67         6
                                               acariasis       0.78      1.00      0.88         7
                                               achalasia       0.65      0.76      0.70        17
                                                    acne       0.70      0.82      0.75  

In [9]:
print(type(model))
print(model)

<class 'sklearn.linear_model._logistic.LogisticRegression'>
LogisticRegression(max_iter=5000)


In [10]:
# Save the Model
import joblib

# Save trained model
joblib.dump(model, "../backend/ml_models/triage_model.pkl")

# Save feature column order
joblib.dump(list(X.columns), "../backend/ml_models/feature_columns.pkl")

print("Model and feature columns saved successfully")

Model and feature columns saved successfully


In [11]:
import joblib

# Verify what was actually saved
loaded = joblib.load("../backend/ml_models/triage_model.pkl")
print("Type of saved object:", type(loaded))
print(loaded)

Type of saved object: <class 'sklearn.linear_model._logistic.LogisticRegression'>
LogisticRegression(max_iter=5000)


In [12]:
# Testing the MOdel
# Load model again (to simulate backend usage)
loaded_model = joblib.load("../backend/ml_models/triage_model.pkl")
feature_columns = joblib.load("../backend/ml_models/feature_columns.pkl")

# Take one real test sample
sample = X_test.iloc[0:1]

prediction = loaded_model.predict(sample)

print("Predicted disease:", prediction[0])
print("Actual disease:", y_test.iloc[0])

Predicted disease: adjustment reaction
Actual disease: adjustment reaction


In [13]:
# Test With Custom Input
# Create zero input with exact training columns
new_patient = pd.DataFrame(
    np.zeros((1, len(feature_columns))),
    columns=feature_columns
)

# Now activate symptoms EXACTLY as in dataset
new_patient["anxiety and nervousness"] = 1
new_patient["depression"] = 1
new_patient["chest tightness"] = 1

prediction = loaded_model.predict(new_patient)

print("Predicted Disease:", prediction[0])

Predicted Disease: panic disorder
